# Packed (scale/offset) NetCDF files

Files whose variables are stored as int16 with `scale_factor`/`add_offset`. pyramids unpacks transparently on read; note the dtype/scale columns in the variable table below.

In [ ]:
%matplotlib inline
from pathlib import Path
import tempfile

import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

from pyramids.netcdf import NetCDF, UgridDataset
from pyramids.feature import FeatureCollection

DATA = Path('../../../../examples/data/netcdf/samples')

## `cf__20v__1d3-3d17.nc`

ECMWF surface fields: 17 packed int16 variables sharing one (time, lat, lon) grid. Longitudes run 0–360.

**Open the file and inspect the container**

In [ ]:
nc = NetCDF.read_file(DATA / 'cf__20v__1d3-3d17.nc')
nc

**Dimensions and variables**

In [ ]:
# get_all_metadata() returns a NetCDFMetadata whose summary lists dimensions and, for each
# variable, its dims / shape / dtype / unit / scale-offset.
meta = nc.get_all_metadata()
print(meta)

**Global attributes**

In [ ]:
nc.global_attributes

**Plot the variable** (a 2-D slice is auto-selected for >2-D variables)

In [ ]:
# select the variable to plot
field = nc.get_variable('tcw')
# this grid's longitudes run 0..360; wrap them to -180..180 so it lines up with the coastline
field = field.wrap_longitude()
# plot in the data's own CRS (lon/lat degrees) — no reprojection
glyph = field.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Wrap longitude to −180–180** — this grid spans 0–360 (Pacific-centred). `wrap_longitude` re-frames it to the −180–180 (Greenwich-centred) convention; we wrap the selected variable, then plot the re-centred map.

In [ ]:
# select the variable to plot
field = nc.get_variable('tcw')
# this grid's longitudes run 0..360; wrap them to -180..180 so it lines up with the coastline
field = field.wrap_longitude()
# plot in the data's own CRS (lon/lat degrees) — no reprojection
glyph = field.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Retrieve the underlying data**

In [ ]:
var = nc.get_variable('tcw')
data = var.read_array()
print('shape:', data.shape)
print('min / mean / max:', float(np.nanmin(data)), float(np.nanmean(data)), float(np.nanmax(data)))

**Reduce a dimension** — collapse the time axis to its mean

In [ ]:
time_mean = nc.reduce('time', how='mean')
print('dimensions after reducing time:', dict(time_mean.dimension_sizes))

**Add a variable** — derive a 2-D field and append it as a new variable

In [ ]:
work = Path(tempfile.mkdtemp())
slice2d = data[tuple(0 for _ in range(data.ndim - 2))]
NetCDF.create_from_array(
    arr=slice2d, geo=var.geotransform, epsg=var.epsg or 4326,
    variable_name='tcw_slice0', path=str(work / 'derived.nc'),
)
nc.add_variable(NetCDF.read_file(str(work / 'derived.nc')), 'tcw_slice0')
print('variables after add:', nc.variable_names)

**Remove a variable**

In [ ]:
nc.remove_variable('tcw_slice0')
print('variables after remove:', nc.variable_names)

**Crop with a real-world polygon** — values are stored int16-packed, but `crop` operates on the georeferenced grid all the same. We crop to **South America**. Web Mercator basemaps span −180…180 while this grid is 0–360, so we `wrap_longitude()` the variable and reproject it to Web Mercator (EPSG:3857) — then it lines up with the **OpenStreetMap basemap** (`basemap=True`, fetched over the network). The whole-container crop that follows runs in the file's native 0–360 coordinates (no basemap).

In [ ]:
# South America, in -180..180 longitudes
region = [
    (-81, 9), (-75, 11), (-65, 5), (-50, 5), (-51, -5), (-45, -23),
    (-55, -35), (-66, -40), (-73, -30), (-78, -18), (-81, -5), (-81, 9),
]
aoi = FeatureCollection(gpd.GeoDataFrame(geometry=[Polygon(region)], crs=4326))

# select the variable
tcw = nc.get_variable('tcw')
# this grid is 0..360; wrap to -180..180 to match the polygon
tcw = tcw.wrap_longitude()
# crop to the South America polygon
tcw_sa = tcw.crop(aoi)
print('cropped tcw bounds:', [round(b, 1) for b in tcw_sa.total_bounds])
glyph = tcw_sa.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Crop the whole container** — clip every variable with the same polygon.

In [ ]:
# whole-container crop in the file's native 0-360 longitudes (shift the polygon by +360)
native_region = [(x + 360 if x < 0 else x, y) for x, y in region]
aoi_native = FeatureCollection(gpd.GeoDataFrame(geometry=[Polygon(native_region)], crs=4326))
cropped = nc.crop(aoi_native)
print('variables in cropped container:', cropped.variable_names)
glyph = tcw_sa.plot()
# draw the Natural Earth coastline on top so the geography stays visible over the data
glyph.add_features("coastline", "50m", crs=4326, zorder=5)

**Save the result to a new NetCDF file**

In [ ]:
out = work / 'cropped.nc'
cropped.to_file(out)
print('saved cropped container to', out.name, '->', out.exists())